In [1]:
!pip install requests pandas google-cloud-bigquery db-dtypes

In [2]:
import requests
import pandas as pd
import logging
from datetime import datetime
from google.cloud import bigquery

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [3]:
def fetch_weather_data(city="Chennai", latitude=13.0827, longitude=80.2707, days=7):
    logger.info(f"Fetching weather data for {city}...")

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "daily": ["temperature_2m_max", "temperature_2m_min", "apparent_temperature_max", "apparent_temperature_min", "precipitation_sum", "windspeed_10m_max"],
        "timezone": "Asia/Kolkata",
        "forecast_days": days
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        logger.info("Data fetched successfully.")
        return response.json()
    except requests.exceptions.Timeout:
        logger.error("Request timed out.")
        return None
    except requests.exceptions.RequestException as e:
        logger.error(f"API error: {e}")
        return None

raw_data = fetch_weather_data()
print(raw_data)

{'latitude': 13.110721, 'longitude': 80.2459, 'generationtime_ms': 0.09834766387939453, 'utc_offset_seconds': 19800, 'timezone': 'Asia/Kolkata', 'timezone_abbreviation': 'GMT+5:30', 'elevation': 12.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'apparent_temperature_max': '°C', 'apparent_temperature_min': '°C', 'precipitation_sum': 'mm', 'windspeed_10m_max': 'km/h'}, 'daily': {'time': ['2026-05-29', '2026-05-30', '2026-05-31', '2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04'], 'temperature_2m_max': [39.0, 38.5, 38.5, 37.3, 37.2, 37.2, 36.2], 'temperature_2m_min': [29.9, 29.0, 29.9, 28.3, 28.6, 28.1, 27.2], 'apparent_temperature_max': [44.8, 45.0, 45.4, 43.5, 44.0, 44.5, 42.6], 'apparent_temperature_min': [35.3, 32.0, 33.1, 32.0, 32.8, 33.7, 32.0], 'precipitation_sum': [0.0, 0.1, 0.1, 0.4, 0.6, 2.4, 2.4], 'windspeed_10m_max': [16.6, 14.0, 17.3, 15.4, 14.8, 14.2, 12.8]}}


In [4]:
def transform_weather_data(raw_data, city="Chennai"):
    logger.info("Transforming data...")

    if not raw_data or "daily" not in raw_data:
        logger.error("Invalid or empty data received.")
        return None

    daily = raw_data["daily"]

    df = pd.DataFrame({
        "date": daily["time"],
        "city": city,
        "temp_max_c": daily["temperature_2m_max"],
        "temp_min_c": daily["temperature_2m_min"],
        "apparent_temp_max_c": daily["apparent_temperature_max"],
        "apparent_temp_min_c": daily["apparent_temperature_min"],
        "precipitation_mm": daily["precipitation_sum"],
        "windspeed_max_kmh": daily["windspeed_10m_max"]
    })

    # Handle nulls
    df = df.fillna(0)

    # Derived fields
    df["temp_range_c"] = df["temp_max_c"] - df["temp_min_c"]
    df["feels_like_delta"] = df["apparent_temp_max_c"] - df["temp_max_c"]
    df["heat_stress"] = df["feels_like_delta"].apply(
        lambda x: "High" if x > 3 else ("Moderate" if x > 1 else "Low")
    )
    df["ingested_at"] = datetime.utcnow().isoformat()

    logger.info(f"Transformed {len(df)} rows successfully.")
    return df

df = transform_weather_data(raw_data)
print(df)

         date     city  temp_max_c  temp_min_c  apparent_temp_max_c  \
0  2026-05-29  Chennai        39.0        29.9                 44.8   
1  2026-05-30  Chennai        38.5        29.0                 45.0   
2  2026-05-31  Chennai        38.5        29.9                 45.4   
3  2026-06-01  Chennai        37.3        28.3                 43.5   
4  2026-06-02  Chennai        37.2        28.6                 44.0   
5  2026-06-03  Chennai        37.2        28.1                 44.5   
6  2026-06-04  Chennai        36.2        27.2                 42.6   

   apparent_temp_min_c  precipitation_mm  windspeed_max_kmh  temp_range_c  \
0                 35.3               0.0               16.6           9.1   
1                 32.0               0.1               14.0           9.5   
2                 33.1               0.1               17.3           8.6   
3                 32.0               0.4               15.4           9.0   
4                 32.8               0.6      

/tmp/ipykernel_968/415359962.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df["ingested_at"] = datetime.utcnow().isoformat()


In [5]:
from google.colab import auth
auth.authenticate_user()
print("Authenticated successfully.")

Authenticated successfully.


In [6]:
def load_to_bigquery(df, project_id, dataset_id, table_id):
    logger.info(f"Loading data to BigQuery: {project_id}.{dataset_id}.{table_id}")

    client = bigquery.Client(project=project_id)

    # Create dataset if it doesn't exist
    dataset_ref = bigquery.Dataset(f"{project_id}.{dataset_id}")
    dataset_ref.location = "US"
    try:
        client.create_dataset(dataset_ref, exists_ok=True)
        logger.info(f"Dataset {dataset_id} ready.")
    except Exception as e:
        logger.error(f"Dataset creation error: {e}")
        return False

    # Define schema
    schema = [
        bigquery.SchemaField("date", "DATE"),
        bigquery.SchemaField("city", "STRING"),
        bigquery.SchemaField("temp_max_c", "FLOAT"),
        bigquery.SchemaField("temp_min_c", "FLOAT"),
        bigquery.SchemaField("apparent_temp_max_c", "FLOAT"),
        bigquery.SchemaField("apparent_temp_min_c", "FLOAT"),
        bigquery.SchemaField("precipitation_mm", "FLOAT"),
        bigquery.SchemaField("windspeed_max_kmh", "FLOAT"),
        bigquery.SchemaField("temp_range_c", "FLOAT"),
        bigquery.SchemaField("feels_like_delta", "FLOAT"),
        bigquery.SchemaField("heat_stress", "STRING"),
        bigquery.SchemaField("ingested_at", "STRING"),
    ]

    table_ref = f"{project_id}.{dataset_id}.{table_id}"
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition="WRITE_TRUNCATE"
    )

    try:
        df["date"] = pd.to_datetime(df["date"]).dt.date
        job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
        job.result()
        logger.info(f"Loaded {len(df)} rows into {table_ref}")
        return True
    except Exception as e:
        logger.error(f"Load error: {e}")
        return False

# REPLACE with your actual project ID
PROJECT_ID = "carbide-cairn-357503"
DATASET_ID = "weather_pipeline"
TABLE_ID = "chennai_forecast"

load_to_bigquery(df, PROJECT_ID, DATASET_ID, TABLE_ID)

True

In [7]:
client = bigquery.Client(project=PROJECT_ID)
query = f"""
    SELECT date, city, temp_max_c, temp_min_c, heat_stress, feels_like_delta
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    ORDER BY date
"""
result = client.query(query).to_dataframe()
print(result)

         date     city  temp_max_c  temp_min_c heat_stress  feels_like_delta
0  2026-05-29  Chennai        39.0        29.9        High               5.8
1  2026-05-30  Chennai        38.5        29.0        High               6.5
2  2026-05-31  Chennai        38.5        29.9        High               6.9
3  2026-06-01  Chennai        37.3        28.3        High               6.2
4  2026-06-02  Chennai        37.2        28.6        High               6.8
5  2026-06-03  Chennai        37.2        28.1        High               7.3
6  2026-06-04  Chennai        36.2        27.2        High               6.4
